# Simple Validator Training Chute Demo

This notebook mirrors the semi-live walkthrough: build the validator-training Chute, author a minimal miner submission, execute the sandbox trainer, inspect artifacts, and optionally register the run with the platform API.

## 1. Environment Bootstrap

Clone the repository and install the dependencies that power the validator sandbox and the Chute builder.

In [ ]:
!git clone https://github.com/tensorlink-dev/epochor.git
%cd epochor
!pip install -r requirements.txt
!pip install chutes bittensor fastapi uvicorn apscheduler datasets

Set any required secrets (e.g., Hugging Face token) ahead of time. The sandbox reads the token from `HF_WRITE_TOKEN_ENV` (defaults to `HF_TOKEN`).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_...'  # replace with a write-enabled token

## 2. Build and Inspect the Validator-Training Chute

Construct the Chute image with representative overrides and inspect its configuration.

In [ ]:
from templates.validator_training.template_builder import build_validator_training_template

chute = build_validator_training_template(
    username='demo-handle',
    gpu_count=1,
    min_vram_gb_per_gpu=12,
    concurrency=1,
    timeout_seconds=1800,
    extra_pip=['wandb==0.17.0'],
)

print(chute.image.python_version)
print(chute.image.pip_packages)
print(chute.entry_file, chute.entry_point)
print(chute.environment)

## 3. Generate a Tiny Regression Dataset

Create a repeatable dataset that the sandbox can ingest. The trainer accepts `cfg["dataset_path"]` pointing to a `.npz` or `.pt` file with `x` and `y` arrays, so you can simulate a richer task than the built-in toy regression.


In [ ]:
%%bash
mkdir -p demo_submission/data
python - <<'PY'
import numpy as np
from pathlib import Path

rng = np.random.default_rng(1234)
n_samples, d_in = 2048, 16
x = rng.normal(size=(n_samples, d_in)).astype('float32')
weights = rng.normal(size=(d_in, 32)).astype('float32')
hidden = np.tanh(x @ weights)
target_w = rng.normal(size=(32, 1)).astype('float32')
y = hidden @ target_w + 0.05 * rng.normal(size=(n_samples, 1)).astype('float32')

out_path = Path('demo_submission/data/regression_dataset.npz')
out_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(out_path, x=x, y=y)
print(f'saved dataset to {out_path}')
PY


If you want to validate the Hugging Face streaming path instead of the handcrafted dataset, swap the configuration dictionary with the snippet below. It streams a budgeted number of windows from `tensorlink-dev/gifteval-iid` and exercises the windowed timeseries helper in the sandbox.

```python
cfg = {
    'seed': 1234,
    'train_batch_size': 16,
    'max_steps': 25,
    'miner_hotkey': 'demo-hotkey',
    'hf_repo_namespace': 'demo-hotkey',
    'hf_repo_name': 'validator-demo',
    'output_tag': 'demo',
    'hf_dataset_repo': 'tensorlink-dev/gifteval-iid',
    'hf_dataset_split': 'train',
    'hf_target_key': 'target',
    'hf_window_stream': {
        'context_length': 128,
        'forecast_horizon': 32,
        'max_batches': 32,
        'budget_batch_size': 16,
        'total_shards': 64,
        'active_shards': 2,
        'streams_per_shard': 2,
        'sample_fraction': 0.5,
    },
}
```

The remainder of the notebook stays the same; the trainer automatically detects the `hf_dataset_repo` and `hf_window_stream` keys and switches into streaming mode.


## 4. Author a Miner Submission with a Small MLP

Create `demo_submission/miner.py` implementing `MinerSubmissionProtocol`. The model below builds a two-layer MLP with ReLU activations and trains with Adam.


In [ ]:
%%bash
mkdir -p demo_submission
cat <<'PY' > demo_submission/miner.py
from templates.validator_training import MinerSubmissionProtocol
from torch import nn
import torch.optim as optim

class Submission(MinerSubmissionProtocol):
    def build_model(self, cfg):
        d_in = int(cfg.get('input_dim', 16))
        hidden = int(cfg.get('hidden_dim', 64))
        return nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def build_optimizer(self, model, cfg):
        lr = float(cfg.get('lr', 1e-3))
        return optim.Adam(model.parameters(), lr=lr)

    def train_step(self, model, batch, optimizer, step_idx, cfg):
        model.train()
        optimizer.zero_grad()
        preds = model(batch['x']).squeeze(-1)
        loss = (preds - batch['y'].squeeze(-1)).pow(2).mean()
        loss.backward()
        optimizer.step()
        return {'loss': loss.item(), 'step': step_idx}
PY


## 5. Run the Sandbox Trainer Locally

Mount the submission and artifacts directories, configure a mock lease, and execute `trainer_entry.run`.


In [ ]:
import asyncio
import os
from pathlib import Path

from templates.validator_training import trainer_entry

submission_dir = Path('demo_submission').resolve()
artifacts_dir = Path('demo_artifacts').resolve()
artifacts_dir.mkdir(exist_ok=True)

os.environ['SUBMISSION_DIR'] = str(submission_dir)
os.environ['ARTIFACTS_DIR'] = str(artifacts_dir)

cfg = {
    'seed': 1234,
    'train_batch_size': 128,
    'input_dim': 16,
    'hidden_dim': 64,
    'dataset_path': 'data/regression_dataset.npz',
    'max_steps': 200,
    'max_seconds': 120,
    'lr': 5e-3,
    'miner_hotkey': 'demo-hotkey',
    'hf_repo_namespace': 'demo-hotkey',
    'hf_repo_name': 'validator-demo',
    'output_tag': 'demo',
}

lease = {
    'submission_id': 'demo-submission',
    'model_id': 'demo-model',
    'round': 1,
}

result = asyncio.run(trainer_entry.run({'cfg': cfg, 'lease': lease}))
print(result)


## 6. Inspect Artifacts

Check the staged checkpoint, metadata, and upload payload.

In [ ]:
list(artifacts_dir.iterdir())

In [ ]:
import json
metadata_files = [p for p in artifacts_dir.glob('**/*.json')]
for meta in metadata_files:
    print(meta)
    with open(meta) as fh:
        data = json.load(fh)
    print(data.keys())

## 7. (Optional) Register with the Platform API

Exercise the FastAPI endpoints using the freshly trained model information.

In [ ]:
from fastapi.testclient import TestClient
from api.config import Settings
from api import database
from api.main import create_app

os.environ['EPOCHOR_DATABASE_URL'] = 'sqlite+pysqlite:///:memory:'
database.configure_engine(os.environ['EPOCHOR_DATABASE_URL'])
database.init_db()
app = create_app()
client = TestClient(app)

settings = Settings()
settings.allowed_miner_hotkeys.append('demo-hotkey')
settings.allowed_validator_hotkeys.append('demo-validator')

submission = client.post('/miner/submit', json={
    'hotkey': 'demo-hotkey',
    'model_code_url': 'https://huggingface.co/demo-hotkey/validator-demo',
}).json()

lease = client.post('/validator/request-training-job', json={
    'validator_hotkey': 'demo-validator',
}).json()

print(submission)
print(lease)